In [1]:
#setup
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def scrape_global_firepower():
    base_url = "https://www.globalfirepower.com/countries-listing.php"
    
    # Check if file exists to avoid crash
    try:
        with open("links_for_military_data.txt", "r") as f:              #scrape additional metrices
            metric_urls = [line.strip() for line in f if line.strip().startswith("http")]
    except:
        metric_urls = []

    print("--- STEP 1: Hunting for Ranks and Scores ---")
    response = requests.get(base_url, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")
    containers = soup.select("div.recordsetContainer")
    
    countries, ranks, scores = [], [], []

    # We use enumerate(containers, 1) to get the position (1, 2, 3...) 
    # as a guaranteed backup for the rank.
    for i, box in enumerate(containers, 1):
        full_text = box.get_text(separator=" ", strip=True)
        
        # 1. Extract Country Name
        name_tag = box.find("span", class_="textUppercase") or box.find("span", class_="textShadow")
        country_name = name_tag.text.strip() if name_tag else "Unknown"

        # 2. Extract Rank (Foolproof method)
        # First, try to find it in the text
        rank_match = re.search(r'Rank[:\s]*(\d+)', full_text, re.IGNORECASE)
        if rank_match:
            rank_val = rank_match.group(1)
        else:
            # If "Rank" word isn't found, use the current loop position 'i'
            # because the list is already in rank order.
            rank_val = str(i)

        # 3. Extract Power Index Score 
        score_match = re.search(r'(?:PwrIndx[:\s]*)(\d+\.\d+)', full_text, re.IGNORECASE)
        score_val = score_match.group(1) if score_match else "0.0"

        # Fallback for score if needed
        if score_val == "0.0":
            decimal_match = re.search(r'(\d\.\d{3,})', full_text)
            if decimal_match: score_val = decimal_match.group(1)

        countries.append(country_name)
        ranks.append(rank_val)
        scores.append(score_val)

    df = pd.DataFrame({
        "Country": countries,
        "pwrind_rank": ranks,
        "pwrind_score": scores
    })
    
    print(f"Captured {len(df)} countries.")
    print("Preview of Rank/Score extraction:")
    print(df[['Country', 'pwrind_rank', 'pwrind_score']].head(5))

    print(f"\n--- STEP 2: Scraping Additional Metrics ---")
    for url in metric_urls:
        if "countries-listing.php" in url: continue
        filename = url.split("/")[-1]
        print(f"Scraping: {filename}")
        
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            s = BeautifulSoup(resp.text, "html.parser")
            rows = s.select("div.recordsetContainer")

            temp_countries, temp_values = [], []
            for row in rows:
                c_tag = row.find("span", class_="textUppercase") or row.find("span", class_="textShadow")
                if c_tag:
                    c_name = c_tag.text.strip()
                    # Values are usually the text in the very last span of the row
                    val_tags = row.find_all("span")
                    if val_tags:
                        val = val_tags[-1].text.strip()
                        temp_countries.append(c_name)
                        temp_values.append(val)

            col_name = filename.replace(".php", "").replace("-", "_")
            temp_df = pd.DataFrame({"Country": temp_countries, col_name: temp_values})
            df = df.merge(temp_df, on="Country", how="left")
            time.sleep(1)
        except Exception as e:
            print(f"Skipping {filename} due to error.")

    print(f"\n--- STEP 3: Final Data Cleaning ---")
    # This regex is very careful: it removes commas but keeps the decimal for the score
    for col in df.columns:
        if col == "Country": continue
        df[col] = df[col].astype(str).str.replace(",", "")
        # Extract number (integer or decimal)
        df[col] = df[col].str.extract(r'(\d+\.?\d*)')[0].fillna("0")

    return df

# Run and Save
df_final = scrape_global_firepower()
df_final.to_csv("military_raw_.csv", index=False)
print("\n✅ DONE! Open 'military_raw_.csv' to see the results.")

--- STEP 1: Hunting for Ranks and Scores ---
Captured 145 countries.
Preview of Rank/Score extraction:
         Country pwrind_rank pwrind_score
0  United States           1       0.0744
1         Russia           2       0.0788
2          China           3       0.0788
3          India           4       0.1184
4    South Korea           5       0.1656

--- STEP 2: Scraping Additional Metrics ---
Scraping: total-population-by-country.php
Scraping: available-military-manpower.php
Scraping: manpower-fit-for-military-service.php
Scraping: manpower-reaching-military-age-annually.php
Scraping: active-military-manpower.php
Scraping: active-reserve-military-manpower.php
Scraping: manpower-paramilitary.php
Scraping: capital-cities-by-total-population.php
Scraping: aircraft-total.php
Scraping: aircraft-total-fighters.php
Scraping: aircraft-total-attack-types.php
Scraping: aircraft-total-transports.php
Scraping: aircraft-total-trainers.php
Scraping: aircraft-total-special-mission.php
Scraping: a